In [ ]:
RUN_OFFICIAL_TEST = False


# FD001 Official Test Final Once

Este notebook queda reservado para el final. Solo evalúa el test oficial si se cambia manualmente `RUN_OFFICIAL_TEST = True`. No debe usarse para elegir hiperparámetros.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    prepare_fd001_temporal_full_train_for_test,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    official_test_prediction_frame,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


In [ ]:
import ast
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

class FlexibleMLPRegressor:
    def __init__(self, hidden_layers, dropout=0.1, lr=1e-3, weight_decay=1e-4, batch_size=256,
                 max_epochs=250, patience=25, validation_fraction=0.15, random_state=42, device=None):
        self.hidden_layers = list(hidden_layers)
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = int(batch_size)
        self.max_epochs = int(max_epochs)
        self.patience = int(patience)
        self.validation_fraction = validation_fraction
        self.random_state = random_state
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_ = None
        self.history_ = []

    def _build_model(self, input_dim):
        layers = []
        prev = input_dim
        for width in self.hidden_layers:
            layers.extend([nn.Linear(prev, int(width)), nn.ReLU()])
            if self.dropout > 0:
                layers.append(nn.Dropout(float(self.dropout)))
            prev = int(width)
        layers.append(nn.Linear(prev, 1))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        set_global_seed(self.random_state)
        X_np = np.asarray(X, dtype=np.float32)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        idx = np.arange(len(X_np))
        train_idx, val_idx = train_test_split(idx, test_size=self.validation_fraction, random_state=self.random_state, shuffle=True)
        X_train = torch.tensor(X_np[train_idx], dtype=torch.float32)
        y_train = torch.tensor(y_np[train_idx], dtype=torch.float32)
        X_val = torch.tensor(X_np[val_idx], dtype=torch.float32).to(self.device)
        y_val = torch.tensor(y_np[val_idx], dtype=torch.float32).to(self.device)
        loader = DataLoader(TensorDataset(X_train, y_train), batch_size=self.batch_size, shuffle=True, generator=torch.Generator().manual_seed(self.random_state))
        self.model_ = self._build_model(X_np.shape[1]).to(self.device)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=float(self.lr), weight_decay=float(self.weight_decay))
        loss_fn = nn.MSELoss()
        best_loss = np.inf
        best_state = None
        patience_left = self.patience
        for epoch in range(1, self.max_epochs + 1):
            self.model_.train()
            losses = []
            for xb, yb in loader:
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                optimizer.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            self.model_.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(self.model_(X_val), y_val).detach().cpu())
            self.history_.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val_loss})
            if val_loss < best_loss - 1e-5:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_left = self.patience
            else:
                patience_left -= 1
            if patience_left <= 0:
                break
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    def predict(self, X):
        if self.model_ is None:
            raise RuntimeError("La MLP debe entrenarse antes de predecir.")
        self.model_.eval()
        X_np = np.asarray(X, dtype=np.float32)
        with torch.no_grad():
            preds = self.model_(torch.tensor(X_np, dtype=torch.float32).to(self.device))
        return np.clip(preds.detach().cpu().numpy().reshape(-1), 0.0, None)


def parse_rul_cap(value):
    if pd.isna(value) or str(value) == "None":
        return None
    return int(float(value))


def parse_window_size(row):
    if "window_size" in row and not pd.isna(row["window_size"]):
        return int(float(row["window_size"]))
    representation = str(row.get("representation", "temporal_w30"))
    if "temporal_w" in representation:
        return int(representation.split("temporal_w")[-1].split("_")[0])
    return DEFAULT_WINDOW_SIZE


def parse_hidden_layers(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return [128, 64, 32]
    return list(ast.literal_eval(str(value)))


def near_failure_weights(y_raw):
    y_raw = np.asarray(y_raw, dtype=float)
    return np.select(
        [y_raw <= 30, (y_raw > 30) & (y_raw <= 60), (y_raw > 60) & (y_raw <= 90)],
        [4.0, 2.0, 1.5],
        default=1.0,
    )


def build_model(model_name, row):
    base_name = str(model_name).replace("_weighted", "").replace("_unweighted", "")
    if base_name.startswith("MLP_tabular_cfg"):
        return FlexibleMLPRegressor(
            hidden_layers=parse_hidden_layers(row.get("hidden_layers", [128, 64, 32])),
            dropout=float(row.get("dropout", 0.1)) if not pd.isna(row.get("dropout", np.nan)) else 0.1,
            lr=float(row.get("lr", 1e-3)) if not pd.isna(row.get("lr", np.nan)) else 1e-3,
            weight_decay=float(row.get("weight_decay", 1e-4)) if not pd.isna(row.get("weight_decay", np.nan)) else 1e-4,
            batch_size=int(float(row.get("batch_size", 256))) if not pd.isna(row.get("batch_size", np.nan)) else 256,
            random_state=SEED,
        )
    factories, _ = available_tabular_factories(random_state=SEED)
    if base_name in factories:
        return factories[base_name]()
    raise ValueError(f"No s? reconstruir el modelo final: {model_name}")


## Ejecución protegida


In [ ]:
if not RUN_OFFICIAL_TEST:
    print("RUN_OFFICIAL_TEST = False. No se entrena ni evalúa test oficial.")
    print("Cuando el candidato final esté cerrado, cambiar manualmente a True y ejecutar este notebook una sola vez.")
else:
    ranking_path = RESULTS_DIR / "fd001_final_validation_ranking.csv"
    if not ranking_path.exists():
        raise FileNotFoundError("Primero ejecutar notebooks de validación y generar fd001_final_validation_ranking.csv")

    ranking = pd.read_csv(ranking_path)
    selected = ranking.iloc[0]
    model_name = selected["model_name"]
    window_size = parse_window_size(selected)
    rul_cap = parse_rul_cap(selected.get("rul_cap", DEFAULT_RUL_CAP))
    representation = f"temporal_w{window_size}"

    print("Candidato final seleccionado solo por validación artificial:")
    display(selected[["source", "representation", "model_name", "rmse", "mae", "dangerous_error_pct"]])

    final_prepared = prepare_fd001_temporal_full_train_for_test(
        data_dir=DATA_DIR,
        max_rul=rul_cap,
        window_size=window_size,
    )
    model = build_model(model_name, selected)
    if str(model_name).endswith("_weighted"):
        model.fit(final_prepared["X_train"], final_prepared["y_train"], sample_weight=near_failure_weights(final_prepared["y_train_raw"]))
    else:
        model.fit(final_prepared["X_train"], final_prepared["y_train"])

    official_predictions = official_test_prediction_frame(final_prepared, model, model_name=model_name, representation=representation)
    official_predictions.to_csv(PREDICTIONS_DIR / "fd001_best_model_v2_predictions.csv", index=False)

    metrics = regression_metrics(official_predictions["y_true_rul_raw"], official_predictions["y_pred_rul"])
    official_metrics = pd.DataFrame([{
        "representation": representation,
        "model_name": model_name,
        "window_size": window_size,
        "rul_cap": "None" if rul_cap is None else rul_cap,
        "n_test": len(official_predictions),
        **metrics,
    }])
    official_metrics.to_csv(RESULTS_DIR / "fd001_best_model_v2_test_metrics.csv", index=False)

    v1_path = RESULTS_DIR / "fd001_official_test_metrics.csv"
    if v1_path.exists():
        v1 = pd.read_csv(v1_path).iloc[0].to_dict()
        v2 = official_metrics.iloc[0].to_dict()
        comparison = pd.DataFrame([
            {"version": "v1", **v1},
            {"version": "v2", **v2},
        ])
        comparison.to_csv(RESULTS_DIR / "fd001_v1_vs_v2_comparison.csv", index=False)
        display(comparison)
    else:
        display(official_metrics)

    print(PREDICTIONS_DIR / "fd001_best_model_v2_predictions.csv")
    print(RESULTS_DIR / "fd001_best_model_v2_test_metrics.csv")
